# Ball Detection — Hough Circles

## Imports

In [ ]:
import os
import cv2
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Load Dataset

In [ ]:
DATASET_DIR = "development_set/"

In [ ]:
image_paths = sorted([
    os.path.join(DATASET_DIR, f)
    for f in os.listdir(DATASET_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Found {len(image_paths)} images")

In [ ]:
def show_images(images, titles=None, max_cols=4, figsize_per_image=(4, 3)):
    n = len(images)
    if n == 0:
        print("No images to display.")
        return

    cols = min(n, max_cols)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_image[0] * cols,
                                                   figsize_per_image[1] * rows))
    axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, str):
                img = cv2.imread(img)

            if img is not None:
                if img.ndim == 2:                          # ← grayscale / mask
                    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
                else:                                      # ← colour image
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            ax.set_title(titles[i] if titles and i < len(titles) else f"Image {i+1}",
                         fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Ball Detection

In [ ]:
def preprocess_lighting(hsv_image):
    # 1. Split the HSV image into its three separate channels
    h, s, v = cv2.split(hsv_image)

    # 2. Create the CLAHE filter
    # clipLimit prevents noise from being amplified too much
    # tileGridSize is the size of the localized "checkerboard" squares
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    # 3. Apply the filter ONLY to the V (brightness) channel
    v_eq = clahe.apply(v)

    # 4. Merge the channels back together
    hsv_eq = cv2.merge((h, s, v_eq))

    return hsv_eq

def get_table_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv = preprocess_lighting(hsv)
    h, w = image.shape[:2]

    # Sample using median to avoid balls in the center
    # 1. Define 5 safe sampling points (Center + 4 inner quadrants)
    points = [
        (h//2, w//2),               # Center
        (int(h*0.4), int(w*0.4)),   # Top-Left inner
        (int(h*0.4), int(w*0.6)),   # Top-Right inner
        (int(h*0.6), int(w*0.4)),   # Bottom-Left inner
        (int(h*0.6), int(w*0.6))    # Bottom-Right inner
    ]

    # 2. Collect a 40x40 patch from ALL 5 locations
    samples = []
    for py, px in points:
        patch = hsv[py-20:py+20, px-20:px+20]
        samples.append(patch)

    # 3. Stack all 8,000 pixels together and find the true median
    all_samples = np.vstack(samples)
    median_hsv = np.median(all_samples, axis=(0, 1))

    # Build tolerance and threshold
    tol = np.array([15, 120, 120])
    lower = np.clip(median_hsv - tol, 0, 255).astype(np.uint8)
    upper = np.clip(median_hsv + tol, 0, 255).astype(np.uint8)

    return cv2.inRange(hsv, lower, upper)

def isolate_largest_blob(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return np.zeros_like(mask)

    largest_contour = max(contours, key=cv2.contourArea)

    clean_mask = np.zeros_like(mask)

    cv2.drawContours(clean_mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

    return clean_mask

In [ ]:
images = [cv2.imread(p) for p in image_paths]
masks  = [get_table_mask(img) for img in images]

show_images(masks, titles=[os.path.basename(p) for p in image_paths],max_cols=3)

In [ ]:
images = [cv2.imread(p) for p in image_paths]
masks  = [get_table_mask(img) for img in images]
blobs  = [isolate_largest_blob(m) for m in masks]

show_images(blobs, titles=[os.path.basename(p) for p in image_paths], max_cols=3)

In [ ]:
def detect_blue_balls(img, existing_circles=None):
    if existing_circles is None:
        existing_circles = []

    table_mask        = get_table_mask(img)
    playing_area_mask = isolate_largest_blob(table_mask)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hsv = preprocess_lighting(hsv)

    h_img, w_img = img.shape[:2]
    pts  = [
        (h_img//2, w_img//2),
        (int(h_img*0.4), int(w_img*0.4)), (int(h_img*0.4), int(w_img*0.6)),
        (int(h_img*0.6), int(w_img*0.4)), (int(h_img*0.6), int(w_img*0.6)),
    ]
    patches   = [hsv[py-20:py+20, px-20:px+20] for py, px in pts]
    table_hsv = np.median(np.vstack(patches), axis=(0, 1))

    if existing_circles:
        median_r = int(np.median([r for _, _, r in existing_circles]))
    else:
        median_r = 18

    r_min = max(8,  int(median_r * 0.70))
    r_max = min(45, int(median_r * 1.35))

    play_img  = cv2.bitwise_and(img, img, mask=playing_area_mask)
    bilateral = cv2.bilateralFilter(play_img, d=9, sigmaColor=75, sigmaSpace=75)
    gray      = cv2.cvtColor(bilateral, cv2.COLOR_BGR2GRAY)

    # Hough Circles setup (param2 dropped slightly to ensure it catches warped/distant balls)
    raw = cv2.HoughCircles(
        gray,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=int(median_r * 1.5),
        param1=50,
        param2=18,
        minRadius=r_min,
        maxRadius=r_max,
    )

    if raw is None:
        return [], []

    candidates = np.round(raw[0]).astype(int)
    blue_circles = []
    blue_bboxes = []

    table_h, table_s, table_v = table_hsv

    for (cx, cy, r) in candidates:
        if playing_area_mask[cy, cx] == 0:
            continue

        overlap = any(
            np.hypot(cx - ex, cy - ey) < (r + er) * 0.6
            for (ex, ey, er) in existing_circles
        )
        if overlap:
            continue

        # Shrink the mask more aggressively to avoid grabbing the table felt around the edges
        inner_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.circle(inner_mask, (cx, cy), max(int(r * 0.75), 3), 255, -1)

        ball_pixels = hsv[inner_mask == 255]
        if len(ball_pixels) < 10:
            continue

        # Separate the channels for fast array math
        h, s, v = ball_pixels[:, 0], ball_pixels[:, 1], ball_pixels[:, 2]

        # 1. SOFTER TABLE FELT REJECTION
        # We only reject it if it's an almost EXACT match to the table, and makes up 60%+ of the circle
        is_table = (np.abs(h.astype(int) - int(table_h)) < 10) & \
                   (np.abs(s.astype(int) - int(table_s)) < 40) & \
                   (np.abs(v.astype(int) - int(table_v)) < 40)
        table_ratio = np.sum(is_table) / len(ball_pixels)

        if table_ratio > 0.60:
            continue

        # 2. FORGIVING BLUE DETECTION (Accounts for glare)
        # We widened the Hue range, and dropped Saturation required to 40
        is_blue = (h >= 85) & (h <= 135) & (s >= 40)
        blue_ratio = np.sum(is_blue) / len(ball_pixels)

        # 3. FORGIVING WHITE/GLARE DETECTION
        # Allows for color bleeding from the felt (higher saturation allowed if it's bright enough)
        is_white_or_glare = (s < 80) & (v > 110)
        white_ratio = np.sum(is_white_or_glare) / len(ball_pixels)

        # --- THE RELAXED ACCEPTANCE LOGIC ---

        # If the ball is almost entirely white (a striped ball facing away),
        # it only needs a TINY speck of blue (like 3%!) to be accepted.
        if white_ratio > 0.40:
            if blue_ratio < 0.03:
                continue
        # If it's a solid ball, it still needs to show *some* blue (at least 15%)
        # amid the shadows and glare.
        elif blue_ratio < 0.15:
            continue

        # Passed!
        final_r = int(r * 1.1)
        blue_circles.append((cx, cy, final_r))
        blue_bboxes.append((cx - final_r, cy - final_r, 2 * final_r, 2 * final_r))

    return blue_circles, blue_bboxes

In [ ]:
processed_images = []
image_titles = []

print(f"Processing {len(image_paths)} images for BLUE balls specifically...")

for path in image_paths:
    img = cv2.imread(path)
    if img is not None:
        # Call the specific blue ball function.
        # Passing None to existing_circles so it looks for all blue balls.
        blue_circles, blue_bboxes = detect_blue_balls(img, existing_circles=None)

        result_img = img.copy()

        # 1. Draw Bounding Boxes (Red in BGR is (0, 0, 255))
        for (x, y, w, h) in blue_bboxes:
            cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 0, 255), 2)

        # 2. Draw Circles (Blue in BGR is (255, 0, 0)) and Center Dots
        for (cx, cy, r) in blue_circles:
            cv2.circle(result_img, (cx, cy), r, (255, 0, 0), 3)
            cv2.circle(result_img, (cx, cy), 1, (0, 255, 0), 2)

        processed_images.append(result_img)
        filename = os.path.basename(path)
        image_titles.append(f"{filename} ({len(blue_circles)} blue)")

# Display the results specifically for blue balls
show_images(processed_images, titles=image_titles, max_cols=3, figsize_per_image=(8, 6))

In [ ]:
def detect_balls(img):
    """
    Receives a BGR image array.
    Returns (final_circles, final_bboxes)
    """
    table_mask = get_table_mask(img)
    playing_area_mask = isolate_largest_blob(table_mask)

    not_table_mask = cv2.bitwise_not(table_mask)
    balls_mask = cv2.bitwise_and(not_table_mask, playing_area_mask)

    heal_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    balls_mask = cv2.morphologyEx(balls_mask, cv2.MORPH_CLOSE, heal_kernel, iterations=1)

    morph_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    balls_mask = cv2.morphologyEx(balls_mask, cv2.MORPH_OPEN, morph_kernel, iterations=1)

    dist_transform = cv2.distanceTransform(balls_mask, cv2.DIST_L2, 5)

    if dist_transform.max() == 0:
        return [], []

    blobs, _ = cv2.findContours(balls_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    circles = []
    bboxes = []

    for blob in blobs:
        blob_mask = np.zeros_like(balls_mask)
        cv2.drawContours(blob_mask, [blob], -1, 255, thickness=cv2.FILLED)

        blob_mask = cv2.morphologyEx(blob_mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))

        blob_dist = np.zeros_like(dist_transform)
        blob_dist[blob_mask == 255] = dist_transform[blob_mask == 255]

        max_val = blob_dist.max()
        if max_val < 5:
            continue

        sharpened_dist = cv2.pow(blob_dist / max_val, 2) * max_val

        area = cv2.contourArea(blob)
        perimeter = cv2.arcLength(blob, True)
        circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

        thresh_ratio = 0.7 if circularity > 0.8 else 0.4

        _, peaks = cv2.threshold(sharpened_dist, thresh_ratio * max_val, 255, 0)
        peaks = np.uint8(peaks)

        peak_contours, _ = cv2.findContours(peaks, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in peak_contours:
            M = cv2.moments(cnt)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                radius = int(dist_transform[cy, cx])

                if 8 < radius < 45:
                    final_r = int(radius * 1.1)
                    circles.append((cx, cy, final_r))
                    bboxes.append((cx - final_r, cy - final_r, 2 * final_r, 2 * final_r))

    # Get blue balls
    blue_circles, blue_bboxes = detect_blue_balls(img, existing_circles=circles)
    circles.extend(blue_circles)
    bboxes.extend(blue_bboxes)

    # FINAL CLEANUP
    final_circles = []
    final_bboxes = []
    ABSOLUTE_MIN_RADIUS = 8
    combined_detections = list(zip(circles, bboxes))
    combined_detections.sort(key=lambda item: item[0][2], reverse=True)

    for (cx, cy, r), bbox in combined_detections:
        if r < ABSOLUTE_MIN_RADIUS:
            continue
        is_overlapping = False
        for (fx, fy, fr) in final_circles:
            distance = np.hypot(cx - fx, cy - fy)
            if distance < (r + fr) * 0.7:
                is_overlapping = True
                break
        if not is_overlapping:
            final_circles.append((cx, cy, r))
            final_bboxes.append(bbox)

    return final_circles, final_bboxes

In [ ]:
# ==========================================
# CELL 1: CALCULATE AND STORE RESULTS
# ==========================================
print("Calculating detections for all images. Please wait...")

# This list will hold all our data so we never have to run detect_balls twice!
master_results = []

for path in image_paths:
    img = cv2.imread(path)
    if img is not None:
        # Calculate BOTH at the exact same time
        circles, bboxes = detect_balls(img)

        # Store all the raw data safely in a dictionary
        master_results.append({
            'filename': os.path.basename(path),
            'original_image': img,
            'circles': circles,
            'bboxes': bboxes
        })

print(f"Done! Successfully processed and memorized {len(master_results)} images.")

In [ ]:
# ==========================================
# CELL 2: VISUALIZE CIRCLES ONLY
# ==========================================
processed_images_circles = []
image_titles_circles = []

for data in master_results:
    # 1. Grab a fresh copy of the original image from memory
    result_img = data['original_image'].copy()

    # 2. Grab the memorized circles
    circles = data['circles']

    # 3. Draw them
    for (cx, cy, r) in circles:
        cv2.circle(result_img, (cx, cy), r, (0, 0, 255), 3)
        cv2.circle(result_img, (cx, cy), 1, (0, 255, 0), 2)

    processed_images_circles.append(result_img)
    image_titles_circles.append(f"{data['filename']} - {len(circles)} Circles")

# Show them instantly
show_images(processed_images_circles, titles=image_titles_circles, max_cols=2, figsize_per_image=(8, 6))

In [ ]:
# ==========================================
# CELL 3: VISUALIZE BOUNDING BOXES ONLY
# ==========================================
processed_images_boxes = []
image_titles_boxes = []

for data in master_results:
    # 1. Grab a fresh copy of the original image from memory
    result_img = data['original_image'].copy()

    # 2. Grab the memorized bounding boxes
    bboxes = data['bboxes']

    # 3. Draw them
    for (x, y, w, h) in bboxes:
        cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 255, 0), 3)

    processed_images_boxes.append(result_img)
    image_titles_boxes.append(f"{data['filename']} - {len(bboxes)} Boxes")

# Show them instantly
show_images(processed_images_boxes, titles=image_titles_boxes, max_cols=2, figsize_per_image=(8, 6))

In [ ]:
def evaluate_detector(csv_path, dataset_dir):
    # 1. Read the CSV file using pandas
    df = pd.read_csv(csv_path, header=None, names=['filename', 'true_count'])

    # Clean up the data
    df = df[df['true_count'].apply(lambda x: str(x).isdigit())]
    df['true_count'] = df['true_count'].astype(int)

    errors = []
    exact_matches = 0
    results = []

    print(f"Evaluating {len(df)} images, please wait...")

    # 2. Loop through the rows
    for index, row in df.iterrows():
        filename = str(row['filename']).strip()
        true_count = row['true_count']

        image_path = os.path.join(dataset_dir, filename)
        if not os.path.exists(image_path):
            print(f"Warning: File not found - {filename}")
            continue

        img = cv2.imread(image_path)
        if img is None:
            print(f"Warning: Could not read image - {filename}")
            continue

        # 3. Predict the number of balls
        circles,bboxes = detect_balls(img)
        pred_count = len(circles)

        # 4. Calculate absolute error for this image
        error = abs(true_count - pred_count)
        errors.append(error)

        if error == 0:
            exact_matches += 1

        results.append({
            'Filename': filename,
            'True Count': true_count,
            'Predicted': pred_count,
            'Error (Missed)': error
        })

    if len(results) == 0:
        print("No valid images processed. Check your CSV path and dataset directory.")
        return

    # 5. Calculate Dataset-Wide Metrics
    results_df = pd.DataFrame(results)

    total_images = len(results_df)
    total_true_balls = results_df['True Count'].sum()
    total_pred_balls = results_df['Predicted'].sum()
    total_absolute_error = results_df['Error (Missed)'].sum()

    # Image-Level Metrics
    mae_per_image = total_absolute_error / total_images
    exact_match_accuracy = (exact_matches / total_images) * 100

    # Entire Dataset Overall Error Rate
    dataset_error_rate = (total_absolute_error / total_true_balls) * 100 if total_true_balls > 0 else 0

    # 6. Print the beautiful summary
    print("\n" + "="*50)
    print("           FULL DATASET EVALUATION")
    print("="*50)
    print(f"Images Evaluated:          {total_images}")
    print(f"Total True Balls:          {total_true_balls}")
    print(f"Total Predicted Balls:     {total_pred_balls}")
    print(f"Total Balls Missed:        {total_absolute_error}")
    print("-" * 50)
    print(f"Image-Level MAE:           {mae_per_image:.3f} (avg balls missed/image)")
    print(f"Exact Match Accuracy:      {exact_match_accuracy:.2f}% (images with 0 errors)")
    print(f"Overall Dataset Error:     {dataset_error_rate:.2f}% (missed / true total)")
    print("="*50 + "\n")

    # 7. Sort the dataframe to show the worst errors
    results_df = results_df.sort_values(by='Error (Missed)', ascending=False).reset_index(drop=True)

    print("--- Top 5 Worst Errors ---")
    display(results_df.head(5))

    return results_df

In [ ]:
my_results = evaluate_detector("ball_count.csv", DATASET_DIR)

In [ ]:
my_results[my_results["Error (Missed)"]>0]

In [ ]:
import matplotlib.image as mpimg

image_name = DATASET_DIR+ "10a_png.rf.bdc9984ba169594ea32b012098ad10dd.jpg"

# 1. Read the image file into an array
img = mpimg.imread(image_name)

# 2. Plot the image
plt.imshow(img)

# 3. Turn off the grid and axes (so it looks like a real photo, not a math graph!)
plt.axis('off')

# 4. Display the image on your screen
plt.show()

# Ball Classification

In [ ]:
# ============================================================
# CELL 1: HSV Color Profiles for all 16 pool balls
# ============================================================
# OpenCV HSV: H 0-179 | S 0-255 | V 0-255
# Red wraps at 0/179 -> needs two ranges
# Stripes 9-15 share hue with solids 1-7 (ball 9 = stripe of 1, etc.)

BALL_HUE_PROFILES = {
    #        hue range(s)              sat_min  val_min  val_max
    1: {'ranges':[(14,38)],            'sat_min':80,'val_min':100,'val_max':255},  # yellow
    2: {'ranges':[(95,125)],           'sat_min':100,'val_min': 60,'val_max':255},  # blue
    3: {'ranges':[(0,6),(165,179)],    'sat_min':150,'val_min': 60,'val_max':255},  # red
    4: {'ranges':[(100,155)],          'sat_min': 70,'val_min': 40,'val_max':255},  # purple
    5: {'ranges':[(5,18)],             'sat_min':90,'val_min':160,'val_max':255},  # orange
    6: {'ranges':[(45,95)],            'sat_min': 90,'val_min': 40,'val_max':255},  # green
    7: {'ranges':[(0,15)],             'sat_min': 70,'val_min': 30,'val_max':150},  # maroon
}
STRIPE_OFFSET = 8   # ball 9 is stripe-yellow (same hue as 1), etc.

BALL_NAMES = {
    0:'cue(white)',  1:'yellow',    2:'blue',    3:'red',
    4:'purple',      5:'orange',    6:'green',   7:'maroon',
    8:'black',       9:'yellow(s)', 10:'blue(s)',11:'red(s)',
    12:'purple(s)', 13:'orange(s)',14:'green(s)',15:'maroon(s)',
}
print("Color profiles loaded.")

In [ ]:
# ============================================================
# CELL 2: ROI extraction + pixel analysis helpers
# ============================================================

def extract_ball_pixels(img, cx, cy, r, inner_ratio=0.82):
    # Returns HSV pixels inside inner circle (inner_ratio*r).
    # Shrinking avoids table-felt contamination at ball edges.
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    inner_r = max(int(r * inner_ratio), 3)
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.circle(mask, (cx, cy), inner_r, 255, -1)
    return hsv[mask == 255]   # shape (N, 3)


def pixel_ratios(pixels):
    # Returns (white_ratio, dark_ratio, colored_pixels)
    # white  : S<55 AND V>160
    # dark   : V<65
    # colored: not white, not dark, S>50
    n = len(pixels)
    if n == 0:
        return 0.0, 0.0, np.empty((0, 3), dtype=np.uint8)
    h, s, v = pixels[:, 0], pixels[:, 1], pixels[:, 2]
    white_m = (s < 80) & (v > 130)
    dark_m  = (v < 65)
    color_m = ~white_m & ~dark_m & (s > 50)
    return white_m.sum() / n, dark_m.sum() / n, pixels[color_m]


def dominant_hue_score(colored_pixels, profile):
    # Fraction of colored pixels matching the profile hue/sat/val ranges.
    if len(colored_pixels) == 0:
        return 0.0
    h, s, v = colored_pixels[:, 0], colored_pixels[:, 1], colored_pixels[:, 2]
    in_rng = np.zeros(len(colored_pixels), dtype=bool)
    for (lo, hi) in profile['ranges']:
        in_rng |= (h >= lo) & (h <= hi)
    in_rng &= (s >= profile['sat_min'])
    in_rng &= (v >= profile['val_min']) & (v <= profile['val_max'])
    return float(in_rng.sum()) / len(colored_pixels)

In [ ]:
# ============================================================
# CELL 3: HSV Histogram Visualizer  (use this to tune thresholds)
# ============================================================

HUE_BANDS = {
    'yellow': ('khaki',       14,  38),
    'blue':   ('dodgerblue',  95, 125),
    'red1':   ('tomato',       0,   6),
    'red2':   ('tomato',     165, 179),
    'purple': ('orchid',     100, 155),
    'orange': ('orange',       5,  15),
    'green':  ('lime',        45,  95),
    'maroon': ('saddlebrown',  0,  15),
}

def visualize_ball_hsv(img, cx, cy, r, title='Ball HSV'):
    pixels = extract_ball_pixels(img, cx, cy, r)
    white_r, dark_r, colored = pixel_ratios(pixels)

    fig, axes = plt.subplots(1, 4, figsize=(18, 3))
    fig.suptitle(
        f'{title}  |  white={white_r:.2f}  dark={dark_r:.2f}  '
        f'colored={len(colored)}/{len(pixels)} px', fontsize=11)

    h_img, w_img = img.shape[:2]; pad = 5
    x1, y1 = max(cx-r-pad, 0), max(cy-r-pad, 0)
    x2, y2 = min(cx+r+pad, w_img), min(cy+r+pad, h_img)
    axes[0].imshow(cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    axes[0].set_title('Crop'); axes[0].axis('off')

    if len(pixels):
        axes[1].hist(pixels[:, 0], bins=90, range=(0,179), color='purple', alpha=0.75)
        axes[1].set_title('Hue (0-179)'); axes[1].set_xlim(0, 179)
        for name, (clr, lo, hi) in HUE_BANDS.items():
            axes[1].axvspan(lo, hi, alpha=0.20, color=clr, label=name)
        axes[1].legend(fontsize=6, loc='upper right')

        axes[2].hist(pixels[:, 1], bins=64, range=(0,255), color='steelblue', alpha=0.75)
        axes[2].set_title('Saturation'); axes[2].set_xlim(0, 255)
        axes[2].axvline(55,  color='red',   ls='--', lw=1, label='white cutoff S<55')
        axes[2].axvline(50,  color='green', ls='--', lw=1, label='color min S>50')
        axes[2].legend(fontsize=7)

        axes[3].hist(pixels[:, 2], bins=64, range=(0,255), color='darkorange', alpha=0.75)
        axes[3].set_title('Value / Brightness'); axes[3].set_xlim(0, 255)
        axes[3].axvline(65,  color='black', ls='--', lw=1, label='dark cutoff V<65')
        axes[3].axvline(160, color='red',   ls='--', lw=1, label='white min V>160')
        axes[3].legend(fontsize=7)

    plt.tight_layout(); plt.show()


def visualize_all_balls_in_image(img, circles, assigned_numbers=None):
    for i, (cx, cy, r) in enumerate(circles):
        if assigned_numbers:
            n = assigned_numbers[i]
            lbl = f'Ball {n} ({BALL_NAMES.get(n, "?")})'
        else:
            lbl = f'#{i}'
        visualize_ball_hsv(img, cx, cy, r, title=lbl)

In [ ]:
# ============================================================
# CELL 4: Per-ball scoring  (v2 — improved cue ball + stripe detection)
# ============================================================
# Key changes:
#   1. Cue ball: primary check white_r>=0.40 (was 0.65).
#      Fallback: if overall median_S < 75 and median_V > 120, treat as cue.
#      This catches cream-colored balls that look white but have slight tint.
#   2. Stripe ramp: activates at white_r>0.05 (was 0.08), reaches 1 at 0.40
#      (was 0.48). Stripe balls with a modest white band now get stripe scores.
#   3. MIN colored pixels guard: skip coloring if almost no colored pixels.

def score_single_ball(img, cx, cy, r):
    pixels = extract_ball_pixels(img, cx, cy, r)
    white_r, dark_r, colored = pixel_ratios(pixels)
    scores = {n: 0.0 for n in range(16)}

    # ── Cue ball (0): primary check ──────────────────────────────────────
    if white_r >= 0.40:
        scores[0] = white_r
        return scores

    # ── Cue ball: median-saturation fallback ─────────────────────────────
    # Catches cream balls that have S=40-70 (just above old white cutoff)
    # but are clearly the lowest-saturation object on the table.
    if len(pixels) >= 10:
        median_s = float(np.median(pixels[:, 1]))
        median_v = float(np.median(pixels[:, 2]))
        if median_s < 75 and median_v > 120:
            # Score scales with how low/desaturated the ball is
            cue_score = (1.0 - median_s / 100.0) * (median_v / 255.0)
            scores[0] = float(np.clip(cue_score, 0.0, 1.0))
            # If this signal is dominant, return early (don't confuse with yellow)
            if scores[0] > 0.35:
                return scores

    # ── 8-ball (dark) ────────────────────────────────────────────────────
    scores[8] = min(dark_r * 1.6, 1.0)

    # ── Colored solids (1-7) and stripes (9-15) ──────────────────────────
    # stripe_lk ramps 0→1 for white_r in [0.05, 0.40]
    stripe_lk = float(np.clip((white_r - 0.10) / 0.18, 0.0, 1.0))
    solid_lk  = 1.0 - stripe_lk

    for solid_num, profile in BALL_HUE_PROFILES.items():
        base = dominant_hue_score(colored, profile)
        if base < 0.04:
            continue
        scores[solid_num]             = base * solid_lk
        scores[solid_num + STRIPE_OFFSET] = base * stripe_lk

    return {k: min(v, 1.0) for k, v in scores.items()}

In [ ]:
# ============================================================
# CELL 5: Global assignment with uniqueness constraint
# ============================================================

def assign_ball_numbers(img, circles):
    # Assigns a unique ball number (0-15) to each detected circle.
    #
    # 1. Build score matrix (n_balls x 16).
    # 2. Sort ALL (score, ball_i, number) triples descending.
    # 3. Greedily assign best pair -> remove ball and number from pool.
    # 4. Balls scoring below MIN_SCORE on everything -> labelled -1.

    MIN_SCORE = 0.05
    n = len(circles)
    if n == 0:
        return []

    score_matrix = np.zeros((n, 16), dtype=float)
    for i, (cx, cy, r) in enumerate(circles):
        for num, sc in score_single_ball(img, cx, cy, r).items():
            score_matrix[i, num] = sc

    available   = set(range(16))
    assignments = [-1] * n
    done_balls  = set()

    candidates = sorted(
        [(score_matrix[bi, num], bi, num)
         for bi in range(n)
         for num in range(16)
         if score_matrix[bi, num] > MIN_SCORE],
        reverse=True
    )

    for score, bi, num in candidates:
        if bi in done_balls or num not in available:
            continue
        assignments[bi] = num
        done_balls.add(bi)
        available.discard(num)

    return assignments

In [ ]:
# ============================================================
# CELL 6: Diagnostic - score table for one image
# ============================================================

def print_score_table(img, circles):
    # Shows top-5 candidate numbers for each detected ball.
    rows = []
    for i, (cx, cy, r) in enumerate(circles):
        s    = score_single_ball(img, cx, cy, r)
        top5 = sorted(s.items(), key=lambda x: -x[1])[:5]
        row  = {'ball_idx': i, 'cx': cx, 'cy': cy, 'r': r}
        for num, sc in top5:
            row[f'#{num}({BALL_NAMES[num]})'] = round(sc, 3)
        rows.append(row)
    display(pd.DataFrame(rows).fillna(''))

In [ ]:
# ============================================================
# CELL 7: Annotated-image renderer  (v2 — much more visible labels)
# ============================================================
# Changes:
#   - Filled rectangle background behind each number label
#   - Font size scales with ball radius (minimum 0.55 instead of 0.4)
#   - Thick white outline + black fill for max contrast
#   - Label box always centered on ball

DRAW_COLOR_BGR = {
    0: (255,255,255), 1: (0,220,220),  2: (220,80,0),   3: (0,0,220),
    4: (180,0,180),   5: (0,140,255),  6: (0,180,0),    7: (60,60,140),
    8: (40,40,40),    9: (180,220,100),10: (220,120,0), 11: (80,80,220),
    12:(200,80,200), 13: (80,180,255),14: (80,220,80),  15:(120,80,180),
}

def draw_classified_image(img, circles, assigned_numbers):
    out = img.copy()
    for (cx, cy, r), num in zip(circles, assigned_numbers):
        ball_color = DRAW_COLOR_BGR.get(num, (200, 200, 200))

        # Bounding box (2px border)
        cv2.rectangle(out, (cx-r, cy-r), (cx+r, cy+r), ball_color, 2)

        label  = str(num) if num >= 0 else '?'
        font   = cv2.FONT_HERSHEY_SIMPLEX
        fs     = max(0.55, r / 18.0)         # larger minimum
        th     = max(2, int(r / 10))          # thicker stroke

        # Measure text to center it
        (tw, t_h), baseline = cv2.getTextSize(label, font, fs, th)
        tx = cx - tw // 2
        ty = cy + t_h // 2

        # Filled background rectangle for contrast
        pad = 3
        cv2.rectangle(out,
                      (tx - pad, ty - t_h - pad),
                      (tx + tw + pad, ty + baseline + pad),
                      (0, 0, 0), cv2.FILLED)

        # White thick outline + colored fill
        cv2.putText(out, label, (tx, ty), font, fs, (255,255,255), th + 3, cv2.LINE_AA)
        cv2.putText(out, label, (tx, ty), font, fs, ball_color,    th,     cv2.LINE_AA)

    return out

In [ ]:
# ============================================================
# CELL 8: Run classification on ALL images in master_results
# ============================================================

print("Classifying balls in all images...")
classified_results = []

for entry in master_results:
    img       = entry['original_image']
    circles   = entry['circles']
    numbers   = assign_ball_numbers(img, circles)
    annotated = draw_classified_image(img, circles, numbers)
    classified_results.append({
        'filename':  entry['filename'],
        'circles':   circles,
        'bboxes':    entry['bboxes'],
        'numbers':   numbers,
        'annotated': annotated,
    })

print(f"Done! Classified {len(classified_results)} images.")

In [ ]:
# ============================================================
# CELL 9: Display all classified images
# ============================================================

show_images(
    [r['annotated'] for r in classified_results],
    titles=[f"{r['filename']}  ({len(r['circles'])} balls)" for r in classified_results],
    max_cols=3,
    figsize_per_image=(8, 6)
)

In [ ]:
# ============================================================
# CELL 10: Summary table
# ============================================================

rows = []
for r in classified_results:
    rows.append({
        'filename':    r['filename'],
        'total_balls': len(r['circles']),
        'identified':  sum(1 for n in r['numbers'] if n >= 0),
        'ball_numbers': sorted(n for n in r['numbers'] if n >= 0),
    })
display(pd.DataFrame(rows))

In [ ]:
# ============================================================
# CELL 11: DEEP-DIVE on one image (histograms + score table)
# Change DEBUG_IDX (0-49) to inspect any image.
# ============================================================

DEBUG_IDX = 21

entry       = master_results[DEBUG_IDX]
img_dbg     = entry['original_image']
circles_dbg = entry['circles']
nums_dbg    = assign_ball_numbers(img_dbg, circles_dbg)

print(f"Image: {entry['filename']}  |  {len(circles_dbg)} balls detected")
print("\n=== Score table (top-5 candidates per ball) ===")
print_score_table(img_dbg, circles_dbg)

print("\n=== HSV histograms for each ball ===")
visualize_all_balls_in_image(img_dbg, circles_dbg, assigned_numbers=nums_dbg)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from IPython.display import clear_output

def annotate_all_images(results_list):
    """
    Loops through ALL images in your results list.
    Returns a list of lists containing your ground truth labels.
    """
    all_ground_truths = []
    total_images = len(results_list)
    
    print("=== Master Manual Annotation Mode ===")
    print("Type the correct ball number (0-15). Type -1 if it's unknown/not a ball.")
    print("Type -99 at any time to SAVE and QUIT.")
    
    for img_idx, entry in enumerate(results_list):
        img = entry['original_image']
        circles = entry['circles']
        image_labels = []
        
        # If no circles were found in this image, just append an empty list and skip
        if circles is None or len(circles) == 0:
            all_ground_truths.append([])
            continue
            
        abort_all = False
        
        for i, (cx, cy, r) in enumerate(circles):
            # 1. Safely crop the ball out of the image
            x1, y1 = max(0, cx - r), max(0, cy - r)
            x2, y2 = min(img.shape[1], cx + r), min(img.shape[0], cy + r)
            crop = img[y1:y2, x1:x2]
            
            # 2. Display the crop with an updated title
            plt.figure(figsize=(2, 2))
            plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            plt.title(f"Image {img_idx+1}/{total_images} | Ball {i+1}/{len(circles)}")
            plt.axis('off')
            plt.show()
            
            # 3. Wait for user input
            while True:
                try:
                    user_input = input(f"Enter number for Ball {i} (or -99 to quit): ")
                    label = int(user_input)
                    
                    if label == -99:
                        abort_all = True
                        break
                    elif -1 <= label <= 15:
                        image_labels.append(label)
                        break
                    else:
                        print("Please enter a number between 0 and 15 (or -1).")
                except ValueError:
                    print("Invalid input. Please enter an integer.")
                    
            # Clear the output so the notebook stays clean
            clear_output(wait=True)
            
            # Break the inner circle loop if user typed -99
            if abort_all:
                break
                
        # Break the outer image loop if user typed -99
        if abort_all:
            print(f"Aborted early! Saved annotations for the {img_idx} fully completed images.")
            break
            
        # Add this image's labels to the master list
        all_ground_truths.append(image_labels)
        
    print("\nAnnotation Complete! Here is your master list of ground truths:")
    print("all_actuals =", all_ground_truths)
    return all_ground_truths

# --- HOW TO USE IT ---
# Pass your entire master_results list into the function
#all_actuals = annotate_all_images(master_results)
all_actuals = [[2, 8, 3, 5, 0, 9, 4, 6, 1, 7], [7, 8, 2, 9, 13, 1, 0, 5, 4, 3, 6, 10, 2], [8, 0], [13, 15, 9, 4, 11, 14, 1, 5, 8, 6, 0, 7, 3, 12, 10, 2], [13, 11, 8, 0, 9, 15, 14, 2, 12, 6, 10], [7, 8, 3, 5, 2, 4, 0, 1, 10], [3, 8, 4, 6, 7, 5, 2, 0, 1], [14, 0, 8], [11, 8, 6, 7, 3, 5, 4, 0, 1, 12, 2], [2, 13, 5, 15, 9, 7, 11, 8, 6, 10, 4, 1, 3, 0, 14], [8, 11, 6, 7, 3, 0, 5, -1, 4, 12, 2, 10], [4, 2, 11, 5, 13, 0, 3, 1, 15, 9, 7, 8, 14, 10, 12, 6], [11, 8, 0, 6, 7, 3, 1, 5, 4, 2], [2, 0, 2, 7, 6, 3, 1, 4, 1], [2, 5, 0, 0, 8], [11, 15, 2, 15, 8, 0, 6, 7, 3, 1, 1, 4], [8, 6, 3, 5, 2, 4, 7, 0, 11, 14, 10], [0, 5, 1, 7, 8, 3, 14, 6, 15, 2, 4], [11, 14, 0, 8, 6, 7], [1, 14, 8, 2, 4, 15, 11, 6, 9, 1, 7, 12], [5, 8, 15, 14, 13, 6, 10, 4, 11, 12, 1, 7, 3, 15, 0, 2], [4, 1, 7, 3, 6, 0, 8, 1], [7, 7, 9, 8, 0, 6, 1, 3, 4, 10, 2], [8, 6, 0, 1, 4], [3, 5, 1, 13, 9, 11, 0, 7, 15, 8, 6, 14, 4], [13, 8, 15, 0, 12, 9], [14, 7, 3, 15, 8, 10, 4, 9, 2, 0, 1, 1], [11, 9, 14, 1, 5, 13, 0, 12, 15, 8, 10], [5, 13, 7, 12, 14, 0, 4, 8, 15, 2], [2, 1, 7, 0, 4, 8, 12, 6], [13, 12, 0, 14, 4, 8, 2], [3, 9, 1, 5, 13, 2, 14, 15, 7, 11, 8, 0, 6, 10, 4], [13, 12, 14, 0, 8, 15, 2], [6, 15, 9, 5, 1, 1, 4, 11, 14, 10, 12, 13], [6, 13, 8, 15, 1, 3, 7, 5, 14, 11, 4, -1], [11, 9, 14, 13, 0, 15, 12, 8, -1], [2, 11, 5, 8, 9, 15, 13, 3, 1, 7, 12, 4, 2, 10], [6, 9, 8, 14, 15, 11, 0, 13, 7, 2], [7, 3, 6, 5, 1, 8, 9, 14, 13, 12, 11, -1, 15, 0, -1, -1, 2], [2, 15, 0, 8, 8], [9, 7, 11, 4, 3, 12, 13, 8, 5, 14, 15, 0, 10, 6, 2], [2, 8, 0], [7, 7, 8, 15, 1, 6, 6, 11, 3, 12, 0, 9, 13, 4, 2], [2, 7, 14, 12, 15, 3, 1, 1, 6, 0, 7, 4], [14, 5, 2, 15, 6, 10, 9, 3, 11, 7, 15, 8, 0, 4], [5, 8, 14, 11, 15, 12, 6, 9, 3, 4, 13, 10, 2, 0], [9, 3, 6, 15, 13, 14, 7, 4, 8, 2, 1, -1], [7, 9, 10, 11, 3, 2, 12, 8, 0, 4, 5, 6, 1], [2, 9, 13, 0, 1, 15, 3, 5, 11, -1, 7, 8, 14, 4, 10, 6], [2, 12, 5, 8, 3, 10, 15, 11, 6, 1, 7, 0, 14]]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_pipeline(y_true, y_pred):
    """
    Compares the actual labels against the predicted labels and generates a report.
    """
    # We include -1 (Unknown/Error) up to 15 (Striped Maroon)
    labels = list(range(-1, 16)) 
    
    print("=== Model Performance Report ===")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    
    # Generate the Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    plt.title('Confusion Matrix: Predicted vs Actual')
    plt.xlabel('Predicted Ball Number (What your code guessed)')
    plt.ylabel('Actual Ball Number (What you manually typed)')
    plt.show()

In [ ]:
# 1. all_actuals is the list you got from the manual annotation script
# It looks something like: [[0, 9, 1], [8, 2, -1], ...]

all_predictions = []

# 2. Run your code ONLY on the images you manually annotated
# (We use len(all_actuals) to make sure we stop exactly where you stopped annotating)
for i in range(len(all_actuals)):
    entry = master_results[i]
    img = entry['original_image']
    circles = entry['circles']
    
    # Run your shiny new assignment function!
    predicted_numbers = assign_ball_numbers(img, circles)
    all_predictions.append(predicted_numbers)

# 3. Flatten the lists. 
# The evaluation function needs one giant list of numbers, not a list of lists.
flat_actuals = [ball for sublist in all_actuals for ball in sublist]
flat_preds = [ball for sublist in all_predictions for ball in sublist]

# 4. Check for length mismatches (just in case!)
if len(flat_actuals) != len(flat_preds):
    print(f"ERROR: You have {len(flat_actuals)} actual labels but {len(flat_preds)} predictions.")
else:
    # 5. RUN THE TEST!
    evaluate_pipeline(flat_actuals, flat_preds)

In [ ]:
import math
import cv2
import matplotlib.pyplot as plt

def visualize_errors(results_list, actuals_list, predictions_list):
    """
    Finds mismatches between actuals and predictions, crops the misclassified 
    balls from the original images, and plots them in a grid.
    """
    error_crops = []
    error_titles = []
    
    # 1. Loop through every image you tested
    for img_idx in range(len(actuals_list)):
        entry = results_list[img_idx]
        img = entry['original_image']
        circles = entry['circles']
        
        actual_labels = actuals_list[img_idx]
        pred_labels = predictions_list[img_idx]
        
        # Check to make sure lists match in length (safety check)
        if len(actual_labels) != len(pred_labels) or len(circles) != len(actual_labels):
            print(f"Skipping Image {img_idx}: Mismatch in number of circles/labels.")
            continue
            
        # 2. Loop through every ball in that specific image
        for ball_idx, (cx, cy, r) in enumerate(circles):
            actual = actual_labels[ball_idx]
            pred = pred_labels[ball_idx]
            
            # 3. IF IT MADE A MISTAKE: Crop it and save it!
            if actual != pred:
                # Safely crop the ball (adding bounds so it doesn't crash on edges)
                x1, y1 = max(0, cx - r), max(0, cy - r)
                x2, y2 = min(img.shape[1], cx + r), min(img.shape[0], cy + r)
                
                crop = img[y1:y2, x1:x2]
                error_crops.append(crop)
                
                # Create a title showing the mistake
               # Create a title showing the exact location of the mistake!
                error_titles.append(f"Img:{img_idx} B:{ball_idx} | T:{actual} P:{pred}")
                
    # 4. Plotting the errors in a nice grid
    if len(error_crops) == 0:
        print("🎉 Perfect! No errors found in this test set.")
        return
        
    print(f"Found {len(error_crops)} misclassified balls. Generating gallery...")
    
    cols = 5
    rows = math.ceil(len(error_crops) / cols)
    
    plt.figure(figsize=(15, 3 * rows))
    
    for i, crop in enumerate(error_crops):
        plt.subplot(rows, cols, i + 1)
        # Convert BGR to RGB so colors look correct in matplotlib
        plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        plt.title(error_titles[i], color='red', fontweight='bold')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

# --- HOW TO RUN IT ---
# Pass in your master image list, your manual labels, and your code's predictions
visualize_errors(master_results, all_actuals, all_predictions)

In [ ]:
# 1. Fix the Blue balls (Solid 2 vs Stripe 10)
all_actuals[13][0]  = 8
all_actuals[13][8] = 5
all_actuals[14][3] = 4
all_actuals[15][3]  = 9
all_actuals[15][10] = 5

# 2. Fix the Maroon balls (Solid 7 vs Stripe 15)
all_actuals[19][0]  = 5
all_actuals[19][8] = 13
all_actuals[20][13]  = 9
all_actuals[21][1]  = 5
all_actuals[22][1] = 5
all_actuals[26][10] = 5
all_actuals[33][5] = 0
all_actuals[42][6] = 5

print("Ground truth patched successfully!")

# 3. Flatten the lists again
flat_actuals = [ball for sublist in all_actuals for ball in sublist]
flat_preds = [ball for sublist in all_predictions for ball in sublist]

# 4. Re-run the evaluation
evaluate_pipeline(flat_actuals, flat_preds)